## RAG application using typesense

In [13]:
import typesense
import os

In [ ]:
from dotenv import load_dotenv

load_dotenv()
typesense_host = os.getenv("TYPESENSE_HOST")
typesense_api_key = os.getenv("TYPESENSE_API_KEY")
if not typesense_host or not typesense_api_key:
    raise ValueError("The host or key is not working properly. Please check your .env file.")

client = typesense.Client(
    {
        "nodes": [
            {
                "host": typesense_host,
                "port": 443,
                "protocol": "https"
            }
        ],
        "api_key": typesense_api_key,
        "connection_timeout_seconds": 2
    }
)

# Create a collection
books_schema = {
    "name": "books",
    "fields": [
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]", "facet": True},
        {"name": "publication_year", "type": "int32"},
        {"name": "ratings_count", "type": "int32"},
        {"name": "average_rating", "type": "float"},
    ],
    "default_sorting_field": "ratings_count"
}

# print(client.collections.create(books_schema))

In [15]:
with open("books.json", "r", encoding="utf-8") as f:
    data = f.read()
    client.collections["books"].documents.import_(data)

In [16]:
search_parameters = {
    "q": "Harry Potter",
    "query_by": "title,authors",
    "sort_by": "ratings_count:desc",
    "per_page": 5,
    "filter_by": "publication_year:>1998"
}
client.collections["books"].documents.search(search_parameters)

{'facet_counts': [],
 'found': 14,
 'hits': [{'document': {'authors': ['J.K. Rowling',
     ' Mary GrandPré',
     ' Rufus Beck'],
    'average_rating': 4.53,
    'id': '18',
    'image_url': 'https://images.gr-assets.com/books/1499277281m/5.jpg',
    'publication_year': 1999,
    'ratings_count': 1832823,
    'title': 'Harry Potter and the Prisoner of Azkaban'},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Prisoner of Azkaban'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Prisoner of Azkaban'}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowli

### Langchain + Typesense + Groq llm + RAG application

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing from the .env file")

In [19]:
loader = TextLoader("text.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6260.81it/s]


In [20]:
docsearch= Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": os.getenv("TYPESENSE_HOST"),
        "port": 443,
        "protocol": "https",
        "typesense_api_key": os.getenv("TYPESENSE_API_KEY"),
        "typesense_collection_name": "langchain"
    },
)

In [21]:
query = "What is Machine Learning?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Machine Learning: The Engine Behind Modern AI
Introduction
Machine Learning (ML) is a core subset of Artificial Intelligence that enables systems to automatically learn and improve from experience without being explicitly programmed. It has become the driving force behind many of the most impressive AI applications we see today.
The Rise of Machine Learning
Although the foundational ideas of machine learning date back to the 1950s and 1960s, the field truly exploded in the last two decades. The combination of massive datasets, powerful GPUs, and better algorithms transformed ML from a research curiosity into a practical technology used by almost every major company.
Key Types of Machine Learning
Machine Learning is generally divided into several major paradigms:
Supervised Learning: The model is trained on labeled data, learning to map inputs to known outputs. Common applications include image classification, spam detection, and price prediction.
Unsupervised Learning: The model discov

In [22]:
### retriever way to god the answer
query = "What is Machine Learning?"
retriever = docsearch.as_retriever()


In [23]:
retriever.invoke(query)[0]

Document(metadata={'source': 'text.txt'}, page_content='Machine Learning: The Engine Behind Modern AI\nIntroduction\nMachine Learning (ML) is a core subset of Artificial Intelligence that enables systems to automatically learn and improve from experience without being explicitly programmed. It has become the driving force behind many of the most impressive AI applications we see today.\nThe Rise of Machine Learning\nAlthough the foundational ideas of machine learning date back to the 1950s and 1960s, the field truly exploded in the last two decades. The combination of massive datasets, powerful GPUs, and better algorithms transformed ML from a research curiosity into a practical technology used by almost every major company.\nKey Types of Machine Learning\nMachine Learning is generally divided into several major paradigms:\nSupervised Learning: The model is trained on labeled data, learning to map inputs to known outputs. Common applications include image classification, spam detection